# Session 2, Module 05: Collections Module


This module covers:
- defaultdict — avoiding KeyError with default factories
- Counter — counting elements, most_common()
- OrderedDict — when insertion order matters
- namedtuple — lightweight immutable objects
- deque — efficient appends/pops from both ends
- ChainMap — searching multiple dicts

Data Engineering Context:
The collections module provides specialized data structures that solve
common problems like counting record types, building frequency tables,
and efficiently processing queues.


In [ ]:
from collections import defaultdict, Counter, OrderedDict, namedtuple, deque, ChainMap

## Defaultdict — Auto-Create Missing Keys


In [ ]:
print("=== defaultdict ===")

# Problem with regular dict: KeyError when key doesn't exist
regular_dict = {}

regular_dict["key"] += 1  # KeyError!
Solution 1: Check first

In [ ]:
if "key" not in regular_dict:
    regular_dict["key"] = 0
regular_dict["key"] += 1

# Solution 2: Use .setdefault()
regular_dict.setdefault("key2", 0)
regular_dict["key2"] += 1

Solution 3 (best): defaultdict
defaultdict(factory) creates missing keys with factory()
Count occurrences

In [ ]:
word_counts = defaultdict(int)  # int() returns 0
words = ["apple", "banana", "apple", "cherry", "banana", "apple"]

for word in words:
    word_counts[word] += 1  # No KeyError, creates 0 if missing

print(f"Word counts: {dict(word_counts)}")

# Group items
records = [
    {"region": "NORTH", "sales": 100},
    {"region": "SOUTH", "sales": 200},
    {"region": "NORTH", "sales": 150},
    {"region": "EAST", "sales": 300},
    {"region": "SOUTH", "sales": 250},
]

# Group sales by region
sales_by_region = defaultdict(list)  # list() returns []
for record in records:
    sales_by_region[record["region"]].append(record["sales"])

print(f"\nSales by region: {dict(sales_by_region)}")

# Calculate totals
totals = {region: sum(sales) for region, sales in sales_by_region.items()}
print(f"Totals: {totals}")

# Nested defaultdict
nested = defaultdict(lambda: defaultdict(int))
nested["user1"]["page_views"] += 1
nested["user1"]["clicks"] += 5
nested["user2"]["page_views"] += 3
print(f"\nNested defaultdict: {dict(nested)}")

## Counter — Counting Made Easy


In [ ]:
print("\n=== Counter ===")

# Create from iterable
status_codes = [200, 200, 404, 200, 500, 404, 200, 301]
code_counts = Counter(status_codes)
print(f"Status codes: {code_counts}")

# Create from string
char_counts = Counter("mississippi")
print(f"Character counts: {char_counts}")

# Access counts
print(f"\nCount of 200: {code_counts[200]}")
print(f"Count of 999 (missing): {code_counts[999]}")  # Returns 0, not KeyError!

# most_common(n) - Get n most common elements
print(f"Most common 2: {code_counts.most_common(2)}")

# Arithmetic operations
counter1 = Counter(a=3, b=2, c=1)
counter2 = Counter(a=1, b=2, c=3)

print(f"\ncounter1 + counter2: {counter1 + counter2}")  # Add counts
print(f"counter1 - counter2: {counter1 - counter2}")  # Subtract (positive only)
print(f"counter1 & counter2: {counter1 & counter2}")  # Min of each
print(f"counter1 | counter2: {counter1 | counter2}")  # Max of each

# Update counter
code_counts.update([200, 200, 503])
print(f"\nAfter update: {code_counts}")

# elements() - Iterator over elements repeated by count
counter = Counter(a=2, b=3)
print(f"elements(): {list(counter.elements())}")

# Practical: Find top error codes
logs = [
    {"status": 200},
    {"status": 404},
    {"status": 500},
    {"status": 200},
    {"status": 404},
    {"status": 200},
    {"status": 500},
    {"status": 503},
]

error_codes = Counter(
    log["status"] for log in logs if log["status"] >= 400
)
print(f"\nTop error codes: {error_codes.most_common(3)}")

## Ordereddict — Ordered Dictionary


In [ ]:
print("\n=== OrderedDict ===")

Note: As of Python 3.7+, regular dict maintains insertion order
OrderedDict is still useful for:
- Explicit intent (code clarity)
- move_to_end() method
- Equality considers order

In [ ]:
od = OrderedDict()
od["first"] = 1
od["second"] = 2
od["third"] = 3
print(f"OrderedDict: {od}")

# move_to_end() - Reorder elements
od.move_to_end("first")  # Move to end
print(f"After move_to_end('first'): {od}")

od.move_to_end("third", last=False)  # Move to beginning
print(f"After move_to_end('third', last=False): {od}")

# popitem() respects order
last_item = od.popitem()  # LIFO (default)
print(f"popitem(): {last_item}, remaining: {od}")

# Practical: LRU-like cache behavior
cache = OrderedDict()
max_size = 3


def access_item(key, value=None):
    """Access item, moving it to end (most recently used)."""
    if key in cache:
        cache.move_to_end(key)
    else:
        if value is not None:
            if len(cache) >= max_size:
                cache.popitem(last=False)  # Remove oldest
            cache[key] = value
    return cache.get(key)


access_item("a", 1)
access_item("b", 2)
access_item("c", 3)
print(f"\nCache: {dict(cache)}")

access_item("a")  # Access 'a', moves to end
print(f"After accessing 'a': {dict(cache)}")

access_item("d", 4)  # Add 'd', removes oldest ('b')
print(f"After adding 'd': {dict(cache)}")

## Namedtuple — Lightweight Classes


In [ ]:
print("\n=== namedtuple ===")

# Define a named tuple type
Customer = namedtuple("Customer", ["id", "name", "email", "status"])

# Create instances
customer1 = Customer(1, "Alice", "alice@example.com", "active")
customer2 = Customer(id=2, name="Bob", email="bob@example.com", status="pending")

print(f"Customer 1: {customer1}")
print(f"Customer 2: {customer2}")

# Access by name (clear!) and index (if needed)
print(f"\ncustomer1.name: {customer1.name}")
print(f"customer1[1]: {customer1[1]}")

Immutable - can't modify
customer1.name = "Alice Smith"  # AttributeError!
Create new with modifications using _replace()

In [ ]:
customer1_updated = customer1._replace(status="inactive")
print(f"Updated: {customer1_updated}")

# Convert to dict
print(f"As dict: {customer1._asdict()}")

# With defaults (Python 3.7+)
Order = namedtuple("Order", ["id", "customer_id", "amount", "status"], defaults=["pending"])
order = Order(1001, 1, 99.99)  # status defaults to "pending"
print(f"\nOrder with default: {order}")

# Practical: Database rows
Row = namedtuple("Row", ["customer_id", "order_count", "total_spent"])
query_results = [
    Row(1, 5, 500.00),
    Row(2, 3, 250.50),
    Row(3, 8, 1200.00),
]

print("\nQuery results:")
for row in query_results:
    print(f"  Customer {row.customer_id}: {row.order_count} orders, ${row.total_spent:.2f}")

# High spenders
high_spenders = [row for row in query_results if row.total_spent > 300]
print(f"High spenders: {[r.customer_id for r in high_spenders]}")

## Deque — Double-Ended Queue


In [ ]:
print("\n=== deque ===")

deque is optimized for O(1) append/pop from both ends
Lists are O(n) for operations at the beginning
Create deque

In [ ]:
dq = deque([1, 2, 3])
print(f"Initial deque: {dq}")

# Operations on right (like list)
dq.append(4)
print(f"After append(4): {dq}")
dq.pop()
print(f"After pop(): {dq}")

# Operations on left (O(1) unlike list!)
dq.appendleft(0)
print(f"After appendleft(0): {dq}")
dq.popleft()
print(f"After popleft(): {dq}")

# Extend from both ends
dq.extend([4, 5])
dq.extendleft([-1, 0])  # Note: order is reversed!
print(f"After extends: {dq}")

# Rotate elements
dq = deque([1, 2, 3, 4, 5])
dq.rotate(2)  # Rotate right by 2
print(f"After rotate(2): {dq}")
dq.rotate(-2)  # Rotate left by 2
print(f"After rotate(-2): {dq}")

Bounded deque (maxlen)
Automatically discards oldest when full

In [ ]:
recent_logs = deque(maxlen=3)
recent_logs.append("log1")
recent_logs.append("log2")
recent_logs.append("log3")
print(f"\nBounded deque (maxlen=3): {recent_logs}")

recent_logs.append("log4")  # Discards log1
print(f"After adding log4: {recent_logs}")

# Practical: Sliding window
def sliding_window_average(values: list, window_size: int) -> list:
    """Calculate moving average using deque."""
    window = deque(maxlen=window_size)
    averages = []

    for value in values:
        window.append(value)
        if len(window) == window_size:
            averages.append(sum(window) / len(window))

    return averages


data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(f"\nMoving average (window=3): {sliding_window_average(data, 3)}")

## Chainmap — Search Multiple Dicts


In [ ]:
print("\n=== ChainMap ===")

ChainMap groups multiple dicts and searches them in order
Changes go to the first dict only

In [ ]:
defaults = {"color": "red", "size": "medium", "debug": False}
user_settings = {"color": "blue"}
cli_args = {"debug": True}

# Search order: cli_args → user_settings → defaults
config = ChainMap(cli_args, user_settings, defaults)

print(f"config['color']: {config['color']}")  # blue (from user_settings)
print(f"config['size']: {config['size']}")    # medium (from defaults)
print(f"config['debug']: {config['debug']}")  # True (from cli_args)

# List all keys
print(f"\nAll keys: {list(config.keys())}")
print(f"All values: {list(config.values())}")

# Access underlying maps
print(f"maps: {config.maps}")

# New child (for temporary overrides)
with_temp = config.new_child({"color": "green"})
print(f"\nWith temp override: {with_temp['color']}")
print(f"Original unchanged: {config['color']}")

## Practical: Complete Example


In [ ]:
print("\n=== Practical: Log Analysis ===")

# Sample log entries
logs = [
    {"timestamp": "2024-01-15 10:00:01", "level": "INFO", "source": "api"},
    {"timestamp": "2024-01-15 10:00:02", "level": "ERROR", "source": "database"},
    {"timestamp": "2024-01-15 10:00:03", "level": "INFO", "source": "api"},
    {"timestamp": "2024-01-15 10:00:04", "level": "WARNING", "source": "api"},
    {"timestamp": "2024-01-15 10:00:05", "level": "ERROR", "source": "api"},
    {"timestamp": "2024-01-15 10:00:06", "level": "INFO", "source": "database"},
    {"timestamp": "2024-01-15 10:00:07", "level": "ERROR", "source": "api"},
]

# Count by level (Counter)
level_counts = Counter(log["level"] for log in logs)
print(f"By level: {level_counts.most_common()}")

# Group by source (defaultdict)
by_source = defaultdict(list)
for log in logs:
    by_source[log["source"]].append(log["level"])
print(f"By source: {dict(by_source)}")

# Recent errors (bounded deque)
recent_errors = deque(maxlen=3)
for log in logs:
    if log["level"] == "ERROR":
        recent_errors.append(log)
print(f"Recent errors: {list(recent_errors)}")

## Summary


In [ ]:
print("\n=== Summary ===")
print("""
defaultdict:
  defaultdict(int)  → Missing keys get 0
  defaultdict(list) → Missing keys get []
  defaultdict(factory) → Calls factory() for missing

Counter:
  Counter(iterable) → Counts elements
  .most_common(n)   → Top n items
  Supports +, -, &, | operations

OrderedDict:
  Maintains insertion order (explicit intent)
  .move_to_end(key) → Reorder elements
  Useful for LRU cache patterns

namedtuple:
  namedtuple("Name", ["field1", "field2"])
  Access by name: obj.field1
  Immutable, tuple-like

deque:
  O(1) append/pop from both ends
  .append(), .appendleft()
  .pop(), .popleft()
  maxlen for bounded queues

ChainMap:
  Groups multiple dicts
  Searches in order
  .new_child() for temp overrides
""")